# Breast Cancer Model Evaluation with scikit-learn

**Topic:** Model Predictions and Evaluations — Basics  
**Goal:** Evaluate model predictions, understand important metrics, and decide which metric fits the breast cancer use case.

We use the built-in scikit-learn breast cancer dataset.

## Learning goals

By the end, you should be able to:

1. Explain what a classification model outputs.
2. Distinguish between a class prediction and a probability score.
3. Read and interpret a confusion matrix.
4. Calculate accuracy, precision, recall/sensitivity, and F1-score.
5. Explain why accuracy alone can be misleading.
6. Change the decision threshold yourself and observe how the confusion matrix changes.
7. Decide which metric should be optimized for this use case.


## 1. Imports

We import the tools needed for the tutorial.


In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


## 2. Load the dataset

The dataset contains numeric measurements of breast tumor samples.

The target has two classes:

- `0` = malignant
- `1` = benign

For this use case, **malignant** is the important class because missing cancer patients is the most serious error.


In [ ]:
data = load_breast_cancer()

X = data.data
y = data.target

print("Target names:", data.target_names)
print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])


## 3. Look at a few rows

Each row is one tumor sample.  
Each column is one measurement.

We only display the first five features to keep the table readable.


In [ ]:
df = pd.DataFrame(X, columns=data.feature_names)
df["target"] = y
df["target_name"] = df["target"].map({0: "malignant", 1: "benign"})

df[[
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean area",
    "mean smoothness",
    "target_name"
]].head()


## 4. Train a simple classification model

We split the data into training and test data.

- The model learns from the training data.
- We evaluate it on the test data.

The goal is not to build the perfect medical model.  
The goal is to understand predictions and evaluation.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=10000))
])

model.fit(X_train, y_train)

print("Model trained.")


## 5. What does the model output?

A classification model can output two things:

| Output | sklearn function | Meaning |
|---|---|---|
| class prediction | `predict()` | final class decision |
| probability score | `predict_proba()` | estimated probability for each class |

For this dataset:

- class `0` = malignant
- class `1` = benign


In [ ]:
# Final class predictions
y_pred = model.predict(X_test)

# Probability scores for both classes
y_proba = model.predict_proba(X_test)

# Probability of the malignant class only
malignant_probability = y_proba[:, 0]

prediction_table = pd.DataFrame({
    "true_label": y_test[:10],
    "true_name": pd.Series(y_test[:10]).map({0: "malignant", 1: "benign"}),
    "probability_malignant": malignant_probability[:10],
    "predicted_label": y_pred[:10],
    "predicted_name": pd.Series(y_pred[:10]).map({0: "malignant", 1: "benign"})
})

prediction_table.round(3)


### Interpretation

Example:

If `probability_malignant = 0.90`, the model thinks the tumor is very likely malignant.

The final class prediction depends on a **decision threshold**.

A simple rule is:

> If probability of malignant is high enough, predict malignant.  
> Otherwise, predict benign.


## 6. Confusion matrix

The confusion matrix compares the actual class with the predicted class.

Rows = actual class  
Columns = predicted class


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

cm_df = pd.DataFrame(
    cm,
    index=["actual malignant", "actual benign"],
    columns=["predicted malignant", "predicted benign"]
)

cm_df


### How to read the confusion matrix

| Cell | Meaning |
|---|---|
| actual malignant, predicted malignant | cancer correctly found |
| actual malignant, predicted benign | cancer missed |
| actual benign, predicted malignant | false alarm |
| actual benign, predicted benign | benign correctly identified |

For this use case, the most dangerous cell is:

> **actual malignant, predicted benign**

That means the model missed a cancer patient.


## 7. Evaluation metrics

Now we calculate the basic metrics.

Important: because malignant is encoded as `0`, we use `pos_label=0`.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision_malignant = precision_score(y_test, y_pred, pos_label=0)
recall_malignant = recall_score(y_test, y_pred, pos_label=0)
f1_malignant = f1_score(y_test, y_pred, pos_label=0)

metrics_df = pd.DataFrame([{
    "accuracy": accuracy,
    "precision_malignant": precision_malignant,
    "recall_sensitivity_malignant": recall_malignant,
    "f1_malignant": f1_malignant
}])

metrics_df.round(3)


### What the metrics mean

| Metric | Question it answers |
|---|---|
| Accuracy | How many predictions were correct overall? |
| Precision | When the model predicts malignant, how often is it correct? |
| Recall / Sensitivity | Of all actual malignant cases, how many did the model find? |
| F1-score | How well does the model balance precision and recall? |

For breast cancer detection, **recall/sensitivity for malignant** is especially important because it tells us how many actual cancer cases were found.


## 8. Threshold experiment

Now the audience can change the threshold.

The model has already produced probability scores.  
The threshold decides when a probability becomes a final class prediction.

Rule:

> If probability of malignant >= threshold, predict malignant.  
> Otherwise, predict benign.

Try different thresholds and observe how the confusion matrix and metrics change.


In [ ]:
def evaluate_threshold(threshold):
    if not 0 <= threshold <= 1:
        raise ValueError("Threshold must be between 0 and 1.")

    y_pred_threshold = np.where(
        malignant_probability >= threshold,
        0,  # predict malignant
        1   # predict benign
    )

    cm = confusion_matrix(y_test, y_pred_threshold, labels=[0, 1])

    cm_df = pd.DataFrame(
        cm,
        index=["actual malignant", "actual benign"],
        columns=["predicted malignant", "predicted benign"]
    )

    missed_cancer_cases = cm[0, 1]
    false_alarms = cm[1, 0]

    metrics_df = pd.DataFrame([{
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred_threshold),
        "precision_malignant": precision_score(y_test, y_pred_threshold, pos_label=0, zero_division=0),
        "recall_sensitivity_malignant": recall_score(y_test, y_pred_threshold, pos_label=0, zero_division=0),
        "f1_malignant": f1_score(y_test, y_pred_threshold, pos_label=0, zero_division=0),
        "missed_cancer_cases": missed_cancer_cases,
        "false_alarms": false_alarms
    }])

    print("Confusion matrix:")
    display(cm_df)

    print("Metrics:")
    display(metrics_df.round(3))


### Audience task

Run the next cell several times and enter different thresholds.

Suggested values:

- `0.7`
- `0.5`
- `0.3`
- `0.1`

Watch especially:

- missed cancer cases
- recall/sensitivity
- false alarms
- precision


In [ ]:
threshold = float(input("Enter a threshold between 0 and 1: "))

evaluate_threshold(threshold)


## 9. Discussion: what should be optimized?

Now stop coding and discuss.

### Question

In breast cancer detection, what should the threshold be optimized for?

### Use-case logic

The main goal is:

> Find all, or almost all, cancer patients.

So we want:

> as few missed cancer cases as possible.

A missed cancer case is a **false negative**:

> actual malignant, predicted benign.

### Metric translation

This means the most important metric is:

> **Recall / sensitivity for malignant**

because recall/sensitivity answers:

> Of all actual malignant tumors, how many did the model find?

### Trade-off

A lower threshold can increase recall and reduce missed cancer cases.  
But it can also increase false alarms and reduce precision.

So the scientific decision is not just:

> Which threshold gives the highest accuracy?

It is:

> Which threshold best matches the consequences of the errors?


## 10. Why accuracy alone can be misleading: class imbalance

Now we look at class distribution and class imbalance.

Class imbalance means:

> One class appears much more often than the other.

If one class is much more common, a model can achieve high accuracy while performing badly on the important minority class.


In [ ]:
df["target_name"].value_counts()


### Simple accuracy trap

Imagine a dataset with:

- 95 benign cases
- 5 malignant cases

A bad model predicts every case as benign.

It gets high accuracy, but it misses every cancer patient.


In [ ]:
y_true_imbalanced = np.array([1] * 95 + [0] * 5)   # 95 benign, 5 malignant
y_pred_all_benign = np.array([1] * 100)            # model predicts all benign

accuracy_bad = accuracy_score(y_true_imbalanced, y_pred_all_benign)
recall_bad = recall_score(
    y_true_imbalanced,
    y_pred_all_benign,
    pos_label=0,
    zero_division=0
)

print("Accuracy:", accuracy_bad)
print("Recall/Sensitivity for malignant:", recall_bad)


### Interpretation

The model has **95% accuracy**, but **0% recall for malignant**.

That means:

> The model missed every cancer patient.

This is why accuracy alone can be misleading, especially when classes are imbalanced or when one error type is much more serious than another.


## Final takeaway

For breast cancer detection:

1. A classification model can output class labels and probability scores.
2. A confusion matrix shows what kinds of mistakes the model makes.
3. Accuracy measures overall correctness, but it can hide dangerous errors.
4. Precision tells us how reliable malignant predictions are.
5. Recall/sensitivity tells us how many actual malignant cases were found.
6. Changing the threshold changes the confusion matrix and metrics.
7. In this use case, the main metric to optimize is **recall/sensitivity for malignant**, while also watching false alarms and precision.

Scientific conclusion:

> The best metric depends on the problem.  
> For breast cancer detection, missing cancer patients is the most serious error, so recall/sensitivity is the key metric.
